# 03 · Hedef Modele (3B) Aktarım ve Çift Hatlı Hizalama (Aşama 3)

**Araştırma sorusu (RQ3).** Vekil modelde keşfedilen $f_{GP}^*$ ile seçilen 1.000 örnek, hedef modelde (Qwen2.5-3B-Base) (a) SFT'de
IFD baseline'ını ve rastgele / kümesiz seçimi geçer mi; (b) tercih optimizasyonunun (GRPO, DPO, R-DPO) başlangıç noktası olarak
daha iyi bir politika sağlar mı?

## 3.1 Deney tasarımı

| Aşama | Modeller | Başlangıç |
|---|---|---|
| SFT (5) | `Model_SFT_Full`, `…_Rand1k`, `…_IFD1k`, `…_GP_Top1k`, `…_GP_Strat1k` | `Qwen2.5-3B` (4-bit NF4 QLoRA) |
| GRPO (2) | `Model_IFD_GRPO`, `Model_GP_GRPO` | `Model_SFT_IFD1k`, `Model_SFT_GP_Strat1k` |
| DPO (2) | `Model_IFD_DPO`, `Model_GP_DPO` | aynı |
| R-DPO (2) | `Model_IFD_RDPO`, `Model_GP_RDPO` | aynı |

SFT: 3 epoch, $\text{lr}=2\times10^{-4}$, LoRA ($r=16,\alpha=32$, tüm doğrusal projeksiyonlar), yalnızca yanıt simgeleri üzerinde kayıp (`completion_only_loss`).

### Hedef fonksiyonlar
**SFT:** $\;\mathcal{L}_{SFT}=-\frac{1}{\lvert y\rvert}\sum_t\log\pi_\theta(y_t\mid x,y_{<t})$

**DPO:** $\;\mathcal{L}_{DPO}=-\mathbb{E}\Big[\log\sigma\Big(\beta\big[\log\tfrac{\pi_\theta(y_w\mid x)}{\pi_{ref}(y_w\mid x)}-\log\tfrac{\pi_\theta(y_l\mid x)}{\pi_{ref}(y_l\mid x)}\big]\Big)\Big]$

**R-DPO (uzunluk düzenlemeli, Park vd. 2024, Eş. 9):**
$$\mathcal{L}_{RDPO}=-\mathbb{E}\Big[\log\sigma\Big(\beta\big[\Delta_w-\Delta_l\big]\;+\;s\,\alpha\,(\lvert y_w\rvert-\lvert y_l\rvert)\Big)\Big],\quad \Delta=\log\tfrac{\pi_\theta}{\pi_{ref}}$$
Orijinal makalede $s=+1$; proje notundaki ifade ($-\alpha(\lvert y_w\rvert-\lvert y_l\rvert)$) için `rdpo_sign=-1` kullanılır. $s=+1$ iken uzun seçilen yanıtlı çiftlerin
gradyan ağırlığı azalır, kısa seçilenlerin artar (uzunluk istismarı baskılanır).

**GRPO:** her istem için $G$ tamamlama örneklenir; grup-göreli avantaj $\hat A_i=\frac{r_i-\operatorname{mean}(r_{1:G})}{\operatorname{std}(r_{1:G})}$,
ödül $r_i=r_\phi(x,y_i)$ **dondurulmuş, 4-bit** `Ray2333/GRM-Llama3-8B-rewardmodel-ft`; KL cezası $\beta_{KL}\,D_{KL}(\pi_\theta\Vert\pi_{ref})$.

### Bellek yönetimi
* Taban model 4-bit NF4 + çift nicemleme; yalnızca LoRA ağırlıkları eğitilir; gradyan checkpointing açık.
* DPO/R-DPO: referans log-olasılıkları **önceden hesaplanır** (`precompute_ref_log_probs`), ayrı referans model yüklenmez.
* GRPO: politika (3B-4bit) + ödül modeli (8B-4bit) birlikte yüklenir; her eğitim sonrası `free_vram()` ile model/eğitici silinir.

In [ ]:
# --- Ortam kurulumu (Colab + yerel) ---------------------------------------------------
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/gp-stratified-sft-selection")      # <- proje klasörünüzün Drive yolu
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["PROJECT_ROOT"] = str(PROJECT)                  # çıktılar PROJECT/outputs altına yazılır
sys.path.insert(0, str(PROJECT / "src"))

%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np, pandas as pd
from IPython.display import display, Markdown
from analysis import set_academic_style, show_figure, show_table
set_academic_style()
print("Proje dizini:", PROJECT)

## 3.2 Yapılandırma
`SMOKE=True` yalnızca çalışma denetimi içindir (çok küçük veri, 3 adım) — **bilimsel sonuç değildir**.

In [ ]:
SMOKE = False
RUN_SFT, RUN_GRPO, RUN_DPO, RUN_RDPO = True, True, True, True
FULL_EPOCHS = 3          # Model_SFT_Full (51.760 örnek) için epoch; süre kısıtında 1 yapılabilir (raporda belirtin)
RDPO_ALPHA, RDPO_SIGN = 0.05, 1

import torch
from alignment import run_alignment, get_args, rdpo_logit_check
kw = dict(limit=64, max_steps=3) if SMOKE else {}
common_kw = dict(full_epochs=FULL_EPOCHS, rdpo_alpha=RDPO_ALPHA, rdpo_sign=RDPO_SIGN, **kw)

hp = pd.Series({k: v for k, v in vars(get_args(**common_kw)).items() if k not in ("stage", "models", "force")}, name="değer").to_frame()
show_table(hp, "tab_hyperparams", "Hedef model eğitimlerinde kullanılan hiperparametreler.", floatfmt="{}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0); print(f"GPU: {p.name}, {p.total_memory/1024**3:.0f} GB")
else:
    print("⚠️ GPU yok — bu defterin eğitim hücreleri GPU gerektirir.")

## 3.3 R-DPO uygulamasının sayısal doğrulaması
TRL'in kayıp fonksiyonunu kopyalamak yerine, uzunluk terimi referans log-olasılıklarına kaydırılarak uygulanır. Aşağıda kaydırmalı kayıp ile tanımdaki kayıp rastgele verilerle karşılaştırılır.

In [ ]:
chk = rdpo_logit_check(beta=0.1, alpha=RDPO_ALPHA, sign=RDPO_SIGN)
show_table(pd.Series(chk, name="değer").to_frame(), "tab_rdpo_check",
           "R-DPO kaydırma hilesinin doğrulanması: 'direct' ve 'shifted' ortalamaları ve en büyük mutlak fark (float32 yuvarlama düzeyinde olmalı).", floatfmt="{:.8f}")
assert chk["max_abs_diff"] < 1e-4, "R-DPO kayıp özdeşliği bozuk!"

## 3.4 Eğitim
Her hücre kesintiye dayanıklıdır: tamamlanmış adaptörler (`outputs/models/<ad>/`) atlanır.

### 3.4.1 SFT (5 model)

In [ ]:
if RUN_SFT:
    run_alignment(stage="sft", **common_kw)

### 3.4.2 Hat 1 — GRPO (dondurulmuş harici ödül modeli ile)

In [ ]:
if RUN_GRPO:
    run_alignment(stage="grpo", **common_kw)

### 3.4.3 Hat 2 — DPO

In [ ]:
if RUN_DPO:
    run_alignment(stage="dpo", **common_kw)

### 3.4.4 Hat 2 — R-DPO

In [ ]:
if RUN_RDPO:
    run_alignment(stage="rdpo", **common_kw)

## 3.5 Bulgular

### 3.5.1 Eğitim kayıp eğrileri

In [ ]:
from analysis import load_logs, plot_curves
from common import SFT_MODELS
logs = load_logs()
show_figure(plot_curves(logs, SFT_MODELS, "loss", "SFT eğitim kaybı"), "fig_sft_loss",
            "Beş SFT modelinin eğitim kaybı (5 adımlık hareketli ortalama). Not: Model_SFT_Full çok daha fazla adım atar; adım eksenleri doğrudan karşılaştırılamaz.")
dpo = ["Model_IFD_DPO", "Model_GP_DPO"]; rdpo = ["Model_IFD_RDPO", "Model_GP_RDPO"]; grpo = ["Model_IFD_GRPO", "Model_GP_GRPO"]
show_figure(plot_curves(logs, dpo + rdpo, "loss", "DPO / R-DPO kaybı"), "fig_pref_loss", "DPO ve R-DPO eğitim kaybı.")
show_figure(plot_curves(logs, dpo + rdpo, "rewards/accuracies", "tercih doğruluğu"), "fig_pref_acc",
            "Örtük ödül doğruluğu $P(\\hat r(y_w)>\\hat r(y_l))$. Not: R-DPO'da raporlanan ödül marjı uzunluk terimini de içerir.")
show_figure(plot_curves(logs, grpo, "reward", "ortalama ödül $r_\\phi$"), "fig_grpo_reward", "GRPO sırasında dondurulmuş ödül modelinin ortalama ödülü.")
show_figure(plot_curves(logs, grpo, "loss", "GRPO kaybı"), "fig_grpo_loss", "GRPO kaybı.")

### 3.5.2 Eğitim maliyeti (süre ve tepe VRAM)

In [ ]:
from analysis import cost_table, plot_cost
ct = cost_table()
show_table(ct, "tab_train_cost", "Eğitim maliyeti: duvar saati süresi ve `torch.cuda.max_memory_allocated` ile ölçülen tepe VRAM.", floatfmt="{:.2f}")
if len(ct):
    show_figure(plot_cost(ct), "fig_train_cost", "Modellere göre tepe VRAM (sol) ve eğitim süresi (sağ, log ölçek).")

## 3.6 Tartışma *(deney sonrası doldurulacak)*
1. SFT kayıp eğrileri: 1.000 örnekli modeller 3 epoch'ta aşırı uyum (overfitting) belirtisi gösteriyor mu? (Eğitim kaybı ve defter 04'teki gerçek değerlendirme ayrışıyor mu?)
2. GRPO ödülü her iki başlangıçta da artıyor mu? GP başlangıçlı politika daha yüksek başlangıç ödülü / daha hızlı artış gösteriyor mu?
3. R-DPO vs DPO: uzunluk düzenlemesi tercih doğruluğunu ne kadar etkiledi? (Defter 04'te yanıt uzunlukları ve LC kazanma oranlarıyla birlikte yorumlayın.)
4. Maliyet: GRPO'nun VRAM ve süre yükü (8B ödül modeli + örnekleme) DPO'ya göre kaç kat?
5. **Tehditler:** tek tohum; GRPO yalnızca 1.000 istem; DPO/R-DPO tek epoch; $\alpha,\beta$ ayarlanmadı.

**Sonraki adım:** `04_evaluation_and_judge`.